# LLM veto distillation in Colab

Runs the whole pipeline on a free Colab CPU (no GPU needed; the models train in seconds).

1. **Setup**: clone and install.
2. **Demo**: synthetic data, no files needed.
3. **Real data** (optional): point at your own snapshot folder on Google Drive. Nothing is uploaded anywhere else.

## 1. Setup

While the repo is private, Colab needs a GitHub token with read access: add it under the key icon (Secrets) as `GH_TOKEN`. Once the repo is public, the token is not needed.

In [ ]:
import os, subprocess
REPO = "balasbrn/llm-veto-distillation"
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("llm-veto-distillation"):
    subprocess.run(["git", "clone", "--depth", "1", url, "llm-veto-distillation"], check=True)
%cd llm-veto-distillation
%pip install -q -e ".[dev,import]"


## 2. Demo on synthetic data

Made-up cards and a hand-written heuristic teacher: it shows the pipeline works, nothing more.

In [ ]:
!python -m pytest -q
!vd demo --out results/synthetic
from IPython.display import Markdown, Image, display
display(Markdown(open("results/synthetic/REPORT.md").read().split("## Figures")[0]))
for f in ["agreement", "r_net", "cumulative_r", "calibration"]:
    display(Image(f"results/synthetic/figures/{f}.png"))


## 3. Real data (optional)

Put your frozen snapshot on Google Drive with this layout:

```
llm-veto-data/h16/cards/<PAIR>/<period>/cards.parquet
llm-veto-data/h16/teacher/<PAIR>-<period>-<run>.jsonl
```

The data stays in your Drive and this Colab session; results are written to `results/real/`. Commit only the report and charts, never the data.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_ROOT = "/content/drive/MyDrive/llm-veto-data"   # change if needed
RUN = "gemini-2.5-flash-v2-think0"                     # or gemini-2.5-flash-v1.1-think1024


In [ ]:
!vd import --data-root "$DATA_ROOT" --source ADA:ADAUSD:dev --source BTC:XBTUSD:dev-long --run "$RUN" --out data/teacher_labels.jsonl
!cat data/teacher_labels.manifest.json
!vd run --data data/teacher_labels.jsonl --out results/real --title "Real data: $RUN teacher"
display(Markdown(open("results/real/REPORT.md").read()))


To keep the results, download `results/real/` (Files panel) or copy it to Drive:

In [ ]:
!cp -r results/real "/content/drive/MyDrive/llm-veto-results-$RUN" 

## 4. Anonymous public sample (optional)

500 cards spread over time; dates become a bar index and group names become G1/G2. Card fields, teacher answers and outcomes are kept. Review the file before committing it as `sample/anonymous_cards.jsonl`.

In [ ]:
!vd sample --data data/teacher_labels.jsonl --n 500 --out sample/anonymous_cards.jsonl
!head -c 600 sample/anonymous_cards.jsonl
!cp sample/anonymous_cards.jsonl /content/drive/MyDrive/anonymous_cards.jsonl